# Struct Type in PySpark

This notebook demonstrates how to work with StructType (nested structures) in PySpark for handling complex hierarchical data.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, struct

spark = SparkSession.builder \
    .appName('Struct Type in PySpark') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## What is StructType?

StructType represents a nested structure (like a nested object) in PySpark. It's similar to a row or record that contains multiple fields, which can themselves be simple types or complex types (including other structs).

**Use Cases:**
- Representing hierarchical data (e.g., person with address)
- Working with JSON data that has nested objects
- Grouping related fields together logically

## Creating DataFrame with Nested Structure

### Example 1: Simple Nested Structure

In [ ]:
# Define a schema with nested structure
schema = StructType([
    StructField("name", StringType(), True),
    StructField("info", StructType([
        StructField("age", IntegerType(), True),
        StructField("address", StringType(), True)
    ]), True)
])

# Sample data - tuples with nested tuples
data = [
    ("John", (30, "1234 Elm St")),
    ("Jane", (25, "5678 Oak St")),
    ("Bob", (35, "9012 Pine St")),
    ("Alice", (28, "3456 Maple Ave"))
]

# Create DataFrame
df = spark.createDataFrame(data, schema)

print("DataFrame with Nested Structure:")
df.show(truncate=False)

print("\nSchema:")
df.printSchema()

## Accessing Nested Fields

### Method 1: Using Dot Notation

In [ ]:
# Access nested fields using dot notation
print("Accessing Nested Fields:")
df.select(
    col("name"),
    col("info.age").alias("age"),
    col("info.address").alias("address")
).show(truncate=False)

### Method 2: Using getField()

In [ ]:
# Alternative: Using getField() method
print("Using getField():")
df.select(
    col("name"),
    col("info").getField("age").alias("age"),
    col("info").getField("address").alias("address")
).show(truncate=False)

### Method 3: Flatten All Fields

In [ ]:
# Flatten the structure - expand nested fields to top level
print("Flattened DataFrame:")
df_flattened = df.select(
    col("name"),
    col("info.age"),
    col("info.address")
)
df_flattened.show(truncate=False)
df_flattened.printSchema()

## Filtering on Nested Fields

In [ ]:
# Filter based on nested field
print("People older than 28:")
df.filter(col("info.age") > 28).show(truncate=False)

# Multiple conditions on nested fields
print("\nPeople aged 25-30:")
df.filter(
    (col("info.age") >= 25) & (col("info.age") <= 30)
).show(truncate=False)

## Creating Nested Structures from Existing Columns

In [ ]:
# Start with flat DataFrame
data_flat = [
    ("John", 30, "1234 Elm St", "New York", "NY"),
    ("Jane", 25, "5678 Oak St", "Los Angeles", "CA"),
    ("Bob", 35, "9012 Pine St", "Chicago", "IL")
]

df_flat = spark.createDataFrame(
    data_flat,
    ["name", "age", "street", "city", "state"]
)

print("Flat DataFrame:")
df_flat.show(truncate=False)

# Create nested structure using struct()
df_nested = df_flat.select(
    col("name"),
    struct(
        col("age"),
        struct(
            col("street"),
            col("city"),
            col("state")
        ).alias("address")
    ).alias("info")
)

print("\nNested DataFrame:")
df_nested.show(truncate=False)
df_nested.printSchema()

## Complex Nested Structure Example

In [ ]:
# Define complex schema: Employee with contact info and job details
complex_schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("contact", StructType([
        StructField("email", StringType(), True),
        StructField("phone", StringType(), True),
        StructField("address", StructType([
            StructField("street", StringType(), True),
            StructField("city", StringType(), True),
            StructField("zipcode", StringType(), True)
        ]), True)
    ]), True),
    StructField("job", StructType([
        StructField("title", StringType(), True),
        StructField("department", StringType(), True),
        StructField("salary", DoubleType(), True)
    ]), True)
])

# Complex nested data
complex_data = [
    (
        1,
        "John Smith",
        ("john@example.com", "555-0101", ("123 Main St", "New York", "10001")),
        ("Software Engineer", "Engineering", 95000.0)
    ),
    (
        2,
        "Jane Doe",
        ("jane@example.com", "555-0102", ("456 Oak Ave", "San Francisco", "94102")),
        ("Data Scientist", "Analytics", 110000.0)
    ),
    (
        3,
        "Bob Johnson",
        ("bob@example.com", "555-0103", ("789 Pine Rd", "Seattle", "98101")),
        ("Product Manager", "Product", 105000.0)
    )
]

df_complex = spark.createDataFrame(complex_data, complex_schema)

print("Complex Nested DataFrame:")
df_complex.show(truncate=False)

print("\nComplex Schema:")
df_complex.printSchema()

## Working with Multi-Level Nested Fields

In [ ]:
# Access deeply nested fields
print("Employee Information with Deeply Nested Fields:")
df_complex.select(
    col("employee_id"),
    col("name"),
    col("contact.email"),
    col("contact.address.city").alias("city"),
    col("job.title").alias("job_title"),
    col("job.salary")
).show(truncate=False)

In [ ]:
# Filter on deeply nested field
print("Employees in New York:")
df_complex.filter(
    col("contact.address.city") == "New York"
).show(truncate=False)

print("\nEmployees with Salary > 100000:")
df_complex.filter(
    col("job.salary") > 100000
).select(
    col("name"),
    col("job.title"),
    col("job.salary")
).show(truncate=False)

## Updating Nested Fields

In [ ]:
from pyspark.sql.functions import when

# Update a nested field (create new struct with updated value)
print("Give 10% raise to all employees:")
df_updated = df_complex.withColumn(
    "job",
    struct(
        col("job.title"),
        col("job.department"),
        (col("job.salary") * 1.10).alias("salary")
    )
)

df_updated.select(
    col("name"),
    col("job.salary").alias("new_salary")
).show()

## Adding New Fields to Struct

In [ ]:
# Add a new field to existing struct
print("Add 'years_of_experience' to job struct:")
df_with_experience = df_complex.withColumn(
    "job",
    struct(
        col("job.title"),
        col("job.department"),
        col("job.salary"),
        when(col("job.title") == "Software Engineer", 5)
        .when(col("job.title") == "Data Scientist", 7)
        .when(col("job.title") == "Product Manager", 8)
        .otherwise(3).alias("years_of_experience")
    )
)

df_with_experience.select("name", "job").show(truncate=False)
df_with_experience.printSchema()

## Working with JSON Data (Natural Struct Format)

In [ ]:
# Create JSON data file
json_data = '''
{"id": 1, "name": "Alice", "profile": {"age": 30, "city": "NYC", "skills": ["Python", "Spark"]}}
{"id": 2, "name": "Bob", "profile": {"age": 25, "city": "LA", "skills": ["Java", "SQL"]}}
{"id": 3, "name": "Charlie", "profile": {"age": 35, "city": "Chicago", "skills": ["Scala", "Hadoop"]}}
'''

with open('/content/people.json', 'w') as f:
    f.write(json_data)

# Read JSON - automatically creates nested structures
df_json = spark.read.json('/content/people.json')

print("JSON Data with Nested Structure:")
df_json.show(truncate=False)
df_json.printSchema()

In [ ]:
# Access nested fields from JSON
print("Accessing Nested Fields from JSON:")
df_json.select(
    col("id"),
    col("name"),
    col("profile.age").alias("age"),
    col("profile.city").alias("city"),
    col("profile.skills").alias("skills")
).show(truncate=False)

## Aggregations with Nested Structures

In [ ]:
from pyspark.sql.functions import avg, count, max as spark_max

# Group by nested field
print("Average Salary by Department:")
df_complex.groupBy("job.department") \
    .agg(
        count("*").alias("employee_count"),
        avg("job.salary").alias("avg_salary"),
        spark_max("job.salary").alias("max_salary")
    ).show()

# Group by deeply nested field
print("\nEmployees by City:")
df_complex.groupBy("contact.address.city") \
    .count() \
    .show()

## Converting Between Struct and Individual Columns

In [ ]:
# Flatten struct to individual columns
print("Flatten Contact Info:")
df_flat_contact = df_complex.select(
    col("employee_id"),
    col("name"),
    col("contact.email"),
    col("contact.phone"),
    col("contact.address.street"),
    col("contact.address.city"),
    col("contact.address.zipcode")
)
df_flat_contact.show(truncate=False)

# Create struct from individual columns
print("\nRecreate Contact Struct:")
df_restructured = df_flat_contact.select(
    col("employee_id"),
    col("name"),
    struct(
        col("email"),
        col("phone"),
        struct(
            col("street"),
            col("city"),
            col("zipcode")
        ).alias("address")
    ).alias("contact")
)
df_restructured.show(truncate=False)
df_restructured.printSchema()

## Practical Example: E-commerce Order

In [ ]:
# Define order schema
order_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer", StructType([
        StructField("name", StringType(), True),
        StructField("email", StringType(), True)
    ]), True),
    StructField("shipping", StructType([
        StructField("address", StringType(), True),
        StructField("city", StringType(), True),
        StructField("country", StringType(), True)
    ]), True),
    StructField("payment", StructType([
        StructField("method", StringType(), True),
        StructField("amount", DoubleType(), True)
    ]), True)
])

# Order data
orders = [
    (1001, ("John Doe", "john@email.com"), ("123 Main St", "NYC", "USA"), ("Credit Card", 299.99)),
    (1002, ("Jane Smith", "jane@email.com"), ("456 Oak Ave", "LA", "USA"), ("PayPal", 149.50)),
    (1003, ("Bob Wilson", "bob@email.com"), ("789 Pine Rd", "Chicago", "USA"), ("Debit Card", 499.99))
]

df_orders = spark.createDataFrame(orders, order_schema)

print("E-commerce Orders:")
df_orders.show(truncate=False)

# Generate invoice format
print("\nInvoice View:")
df_orders.select(
    col("order_id"),
    col("customer.name").alias("customer_name"),
    col("customer.email"),
    col("shipping.city").alias("ship_to_city"),
    col("payment.method").alias("payment_method"),
    col("payment.amount").alias("total_amount")
).show(truncate=False)

## Key Takeaways

### StructType Benefits:
- **Logical Grouping**: Related fields grouped together
- **Schema Clarity**: Clear hierarchical structure
- **JSON Compatibility**: Natural representation of JSON data
- **Type Safety**: Each field has defined type

### Accessing Nested Data:
- **Dot Notation**: `col("struct.field")`
- **getField()**: `col("struct").getField("field")`
- **Multi-level**: `col("struct.nested.field")`

### Operations:
- **Filter**: Can filter on any nested field
- **Select**: Can select nested fields directly
- **Update**: Use `struct()` to recreate with updated values
- **Aggregate**: Group by nested fields

### Best Practices:
- Use structs for logically related data
- Flatten when needed for simpler operations
- Define explicit schemas for complex structures
- Use structs to represent JSON hierarchies naturally

In [ ]:
# Stop Spark Session
spark.stop()